# CoalitionGNN — Veri İndirme (Tam Otomatik)

Tüm 9 kaynak otomatik çekilir. Başarı/başarısızlık özet tablo halinde sona basılır.

**Beklenen toplam süre:** ~15-20 dk (IMF DOTS ülke döngüsü en yavaş kısım).

**Beklenen toplam boyut:** ~500 MB - 1 GB (V-Dem büyük dosya).

## 0. Hazırlık — Drive mount, dizin, paketler

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/coalition_gnn'
RAW_DIR = os.path.join(PROJECT_DIR, 'data/raw')
os.makedirs(RAW_DIR, exist_ok=True)
print(f'Ham veri dizini: {RAW_DIR}')

In [ ]:
!pip install -q requests pandas pyarrow tqdm rpy2

## 1. Yardımcı altyapı — sonuç toplayıcı + indirme fonksiyonları

Her kaynağı bir `try/except` ile sarıp `results` dict'ine bir kayıt ekliyoruz.
Hangi adımın patladığı net görünsün diye.

In [ ]:
import requests
import zipfile
import io
import time
import traceback
from pathlib import Path
from tqdm.auto import tqdm

# Tüm indirme sonuçları burada toplanacak
results = []

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (CoalitionGNN academic research)'
}

def record(source, status, message='', files=None, size_mb=0.0, duration_s=0.0):
    """Sonucu listeye ekle."""
    results.append({
        'source': source,
        'status': status,  # 'OK', 'FAIL', 'PARTIAL'
        'message': message,
        'files': files or [],
        'size_mb': round(size_mb, 1),
        'duration_s': round(duration_s, 1),
    })
    icon = {'OK': '✅', 'FAIL': '❌', 'PARTIAL': '⚠️'}.get(status, '?')
    print(f'{icon} {source}: {status} — {message}')

def dir_size_mb(path):
    """Bir dizinin toplam boyutu MB cinsinden."""
    if not os.path.exists(path):
        return 0.0
    total = sum(f.stat().st_size for f in Path(path).rglob('*') if f.is_file())
    return total / 1e6

def list_files(path):
    """Dizindeki dosya isimlerini listele."""
    if not os.path.exists(path):
        return []
    return sorted([f.name for f in Path(path).rglob('*') if f.is_file()])

def download_file(url, dest_path, headers=None, timeout=120):
    """Tek dosya indir, progress bar ile."""
    headers = headers or HEADERS
    r = requests.get(url, headers=headers, stream=True, timeout=timeout)
    r.raise_for_status()
    total = int(r.headers.get('content-length', 0))
    with open(dest_path, 'wb') as f, tqdm(total=total, unit='B', unit_scale=True,
                                           desc=Path(dest_path).name, leave=False) as pbar:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
            pbar.update(len(chunk))
    return dest_path

def download_and_extract_zip(url, dest_dir, headers=None, timeout=300):
    """ZIP indir, dest_dir altına aç."""
    headers = headers or HEADERS
    os.makedirs(dest_dir, exist_ok=True)
    r = requests.get(url, headers=headers, timeout=timeout)
    r.raise_for_status()
    with zipfile.ZipFile(io.BytesIO(r.content)) as z:
        z.extractall(dest_dir)
    return dest_dir

print('✅ Altyapı hazır')

## 2. COW Veri Setleri

Country codes, NMC (kapasite), MID (çatışma), ittifaklar, ticaret v4.

In [ ]:
# Not: COW WordPress sitesi sürüm güncellemelerinde URL'leri değiştirebiliyor.
# 404 alırsa correlatesofwar.org/data-sets sayfasından güncel linki kopyala.

COW_SOURCES = {
    'cow_country_codes': {
        'url': 'https://correlatesofwar.org/wp-content/uploads/COW-country-codes.csv',
        'type': 'csv',
    },
    'cow_state_system': {
        'url': 'https://correlatesofwar.org/wp-content/uploads/system2016.csv',
        'type': 'csv',
    },
    'cow_nmc': {
        'url': 'https://correlatesofwar.org/wp-content/uploads/NMC_Documentation-6.0.zip',
        'type': 'zip',
        'fallbacks': [
            'https://correlatesofwar.org/wp-content/uploads/NMC_5_0.zip',
            'https://correlatesofwar.org/wp-content/uploads/NMC-60-abridged.zip',
        ],
    },
    'cow_mid': {
        'url': 'https://correlatesofwar.org/wp-content/uploads/MID-5-Data-and-Supporting-Materials.zip',
        'type': 'zip',
        'fallbacks': [
            'https://correlatesofwar.org/wp-content/uploads/MIDB-5.0.csv',
            'https://correlatesofwar.org/wp-content/uploads/MID-5.zip',
        ],
    },
    'cow_alliances': {
        'url': 'https://correlatesofwar.org/wp-content/uploads/version4.1_csv.zip',
        'type': 'zip',
    },
    'cow_trade': {
        'url': 'https://correlatesofwar.org/wp-content/uploads/COW_Trade_4.0.zip',
        'type': 'zip',
    },
}

for name, cfg in COW_SOURCES.items():
    t0 = time.time()
    target = os.path.join(RAW_DIR, 'cow', name)
    os.makedirs(target, exist_ok=True)
    
    # URL listesi: ana URL + fallback'ler
    urls_to_try = [cfg['url']] + cfg.get('fallbacks', [])
    success = False
    last_error = None
    
    for url in urls_to_try:
        try:
            if url.endswith('.zip'):
                download_and_extract_zip(url, target)
            elif url.endswith('.csv'):
                download_file(url, os.path.join(target, os.path.basename(url)))
            else:
                download_file(url, os.path.join(target, os.path.basename(url)))
            record(name, 'OK',
                   message=f"indirildi: {os.path.basename(url)}",
                   files=list_files(target),
                   size_mb=dir_size_mb(target),
                   duration_s=time.time()-t0)
            success = True
            break
        except Exception as e:
            last_error = f'{type(e).__name__}: {str(e)[:120]}'
            continue
    
    if not success:
        record(name, 'FAIL',
               message=last_error,
               duration_s=time.time()-t0)

## 3. ATOP — İttifak Antlaşmaları (süre verisi buradan)

In [ ]:
t0 = time.time()
ATOP_URL = 'http://www.atopdata.org/uploads/6/9/1/3/69134503/atop_5_1__csv_.zip'
atop_dir = os.path.join(RAW_DIR, 'atop')

try:
    download_and_extract_zip(ATOP_URL, atop_dir)
    record('atop', 'OK',
           message='v5.1 indirildi',
           files=list_files(atop_dir),
           size_mb=dir_size_mb(atop_dir),
           duration_s=time.time()-t0)
except Exception as e:
    record('atop', 'FAIL',
           message=f'{type(e).__name__}: {str(e)[:120]}',
           duration_s=time.time()-t0)

## 4. IMF DOTS — Ticaret (yeni SDMX 3.0 API)

Eski IMF JSON REST API (dataservices.imf.org) Haziran 2025'te kapatıldı.
Yeni endpoint: `api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/IMTS/~`

Her ülke çifti için ihracat/ithalat. ~200 ülke × 2 yön.
Rate limit'i aşmamak için arada 0.5s bekliyoruz.

**Not:** Bu adım en yavaş kısım (~30-60 dk). İlk seferde `MAX_COUNTRIES = 10` ile test et.

In [ ]:
t0 = time.time()
imf_dir = os.path.join(RAW_DIR, 'imf_dots')
os.makedirs(imf_dir, exist_ok=True)

import pandas as pd
from io import StringIO

# IMF SDMX 3.0 API — IMTS (eski DOTS)
IMTS_BASE = 'https://api.imf.org/external/sdmx/3.0/data/dataflow/IMF.STA/IMTS/~'

# Büyük ülkelerin ISO-2 kodları (IMF bu API'de ISO-2 kullanıyor)
# Tam listeyi almak için önce codelist'i çekelim
try:
    # Codelist'ten tüm geçerli ülke kodlarını çek
    codelist_url = 'https://api.imf.org/external/sdmx/3.0/structure/codelist/IMF.STA/CL_AREA_IMTS'
    r = requests.get(codelist_url, headers=HEADERS, timeout=60)
    r.raise_for_status()
    
    # JSON formatında gelir
    cl_data = r.json()
    # Codelist'ten ülke kodlarını çıkar
    codes = cl_data.get('data', {}).get('codelists', [{}])[0].get('codes', [])
    valid_countries = [c['id'] for c in codes if len(c['id']) == 2]
    print(f'  IMF IMTS codelist\'inde {len(valid_countries)} ülke kodu var')
    
except Exception as e:
    print(f'  Codelist çekilemedi ({type(e).__name__}), sabit liste kullanılıyor')
    # Fallback: büyük ülkeler (ISO-2)
    valid_countries = [
        'US', 'CN', 'JP', 'DE', 'GB', 'FR', 'IN', 'IT', 'BR', 'CA',
        'RU', 'KR', 'AU', 'ES', 'MX', 'ID', 'NL', 'SA', 'TR', 'CH',
        'PL', 'TH', 'SE', 'BE', 'NG', 'AT', 'NO', 'AE', 'IL', 'IE',
        'SG', 'HK', 'DK', 'PH', 'MY', 'ZA', 'CO', 'EG', 'PK', 'CL',
        'FI', 'BD', 'CZ', 'PT', 'RO', 'NZ', 'GR', 'QA', 'PE', 'IQ',
        'KW', 'HU', 'UA', 'MA', 'SK', 'EC', 'KE', 'DO', 'GT', 'OM',
        'BG', 'LK', 'UZ', 'HR', 'LT', 'TN', 'GH', 'RS', 'SI', 'JO',
        'AZ', 'PA', 'BY', 'TZ', 'LV', 'UY', 'BO', 'PY', 'BH', 'EE',
        'GE', 'SN', 'CI', 'CM', 'TT', 'HN', 'NP', 'IS', 'SV', 'NI',
        'CR', 'LB', 'ZM', 'BW', 'MU', 'AL', 'MN', 'MW', 'AM', 'KG',
    ]

MAX_COUNTRIES = None  # Test için 10 yap; tam toplama için None
countries_to_fetch = valid_countries[:MAX_COUNTRIES] if MAX_COUNTRIES else valid_countries

def fetch_imts_country(country_code, indicator, start_year=1946, end_year=2016):
    """Tek ülke için IMTS verisini CSV olarak çek."""
    # Key format: REF_AREA.INDICATOR.COUNTERPART_AREA.FREQ
    # W00 = World, tüm partnerler
    key = f'{country_code}.{indicator}..A'
    url = f'{IMTS_BASE}/{key}'
    params = {
        'startPeriod': str(start_year),
        'endPeriod': str(end_year),
        'format': 'csvdata',
    }
    r = requests.get(url, params=params, headers=HEADERS, timeout=120)
    r.raise_for_status()
    if len(r.text.strip()) < 10:
        return pd.DataFrame()
    return pd.read_csv(StringIO(r.text))

try:
    # --- İHRACAT ---
    print('--- İhracat çekiliyor ---')
    export_rows = []
    export_failed = []
    
    for country in tqdm(countries_to_fetch, desc='IMF IMTS İhracat'):
        try:
            df = fetch_imts_country(country, 'XG_FOB_USD')
            if len(df) > 0:
                df['reporter'] = country
                df['flow'] = 'export'
                export_rows.append(df)
            time.sleep(0.5)
        except Exception as e:
            export_failed.append((country, str(e)[:80]))
    
    if export_rows:
        export_df = pd.concat(export_rows, ignore_index=True)
        export_df.to_parquet(os.path.join(imf_dir, 'dots_exports.parquet'))
        print(f'  İhracat: {len(countries_to_fetch)-len(export_failed)}/{len(countries_to_fetch)} ülke OK')
    
    # --- İTHALAT ---
    print('\n--- İthalat çekiliyor ---')
    import_rows = []
    import_failed = []
    
    for country in tqdm(countries_to_fetch, desc='IMF IMTS İthalat'):
        try:
            df = fetch_imts_country(country, 'MG_CIF_USD')
            if len(df) > 0:
                df['reporter'] = country
                df['flow'] = 'import'
                import_rows.append(df)
            time.sleep(0.5)
        except Exception as e:
            import_failed.append((country, str(e)[:80]))
    
    if import_rows:
        import_df = pd.concat(import_rows, ignore_index=True)
        import_df.to_parquet(os.path.join(imf_dir, 'dots_imports.parquet'))
        print(f'  İthalat: {len(countries_to_fetch)-len(import_failed)}/{len(countries_to_fetch)} ülke OK')
    
    # --- SONUÇ ---
    all_failed = export_failed + import_failed
    if export_rows or import_rows:
        status = 'OK' if not all_failed else 'PARTIAL'
        msg = f'ihracat: {len(countries_to_fetch)-len(export_failed)}, ithalat: {len(countries_to_fetch)-len(import_failed)} ülke OK'
        if all_failed:
            msg += f' (toplam {len(all_failed)} başarısız istek)'
        record('imf_dots', status,
               message=msg,
               files=list_files(imf_dir),
               size_mb=dir_size_mb(imf_dir),
               duration_s=time.time()-t0)
    else:
        record('imf_dots', 'FAIL',
               message='Hiç veri çekilemedi',
               duration_s=time.time()-t0)

except Exception as e:
    record('imf_dots', 'FAIL',
           message=f'{type(e).__name__}: {str(e)[:120]}',
           duration_s=time.time()-t0)

## 5. UN Voting — Voeten Harvard Dataverse

In [ ]:
t0 = time.time()
un_dir = os.path.join(RAW_DIR, 'un_voting')
os.makedirs(un_dir, exist_ok=True)

DATAVERSE_API = 'https://dataverse.harvard.edu/api'
DOI = 'doi:10.7910/DVN/LEJUQZ'

# Hangi dosya isimlerini istiyoruz (kısmi eşleşme)
WANTED = ['UNVotes', 'Idealpoint', 'AgreementScores']

try:
    r = requests.get(f'{DATAVERSE_API}/datasets/:persistentId/?persistentId={DOI}',
                     timeout=60)
    r.raise_for_status()
    ds = r.json()
    files = ds['data']['latestVersion']['files']
    
    downloaded = []
    skipped = []
    for f in files:
        fname = f['dataFile']['filename']
        fid = f['dataFile']['id']
        if any(w in fname for w in WANTED):
            try:
                url = f'{DATAVERSE_API}/access/datafile/{fid}'
                download_file(url, os.path.join(un_dir, fname))
                downloaded.append(fname)
            except Exception as e:
                skipped.append((fname, str(e)[:80]))
    
    if downloaded:
        status = 'OK' if not skipped else 'PARTIAL'
        msg = f'{len(downloaded)} dosya indirildi'
        if skipped:
            msg += f', {len(skipped)} atlandı'
        record('un_voting', status,
               message=msg,
               files=list_files(un_dir),
               size_mb=dir_size_mb(un_dir),
               duration_s=time.time()-t0)
    else:
        record('un_voting', 'FAIL',
               message='Eşleşen dosya bulunamadı',
               duration_s=time.time()-t0)
except Exception as e:
    record('un_voting', 'FAIL',
           message=f'{type(e).__name__}: {str(e)[:120]}',
           duration_s=time.time()-t0)

## 6. UCDP — Modern çatışma verileri

In [ ]:
t0 = time.time()
ucdp_dir = os.path.join(RAW_DIR, 'ucdp')
os.makedirs(ucdp_dir, exist_ok=True)

# UCDP'nin birden fazla URL formatı denenir — biri çalışırsa devam eder
UCDP_CANDIDATES = [
    'https://ucdp.uu.se/downloads/ucdpprioacd/ucdp-prio-acd-241-csv.zip',
    'https://ucdp.uu.se/downloads/ucdpprioacd/ucdp-prio-acd-241.csv',
    'https://ucdp.uu.se/downloads/ucdpprioacd/ucdp-prio-acd-24.csv',
]

success_url = None
last_error = None
for url in UCDP_CANDIDATES:
    try:
        if url.endswith('.zip'):
            download_and_extract_zip(url, ucdp_dir)
        else:
            download_file(url, os.path.join(ucdp_dir, os.path.basename(url)))
        success_url = url
        break
    except Exception as e:
        last_error = f'{type(e).__name__}: {str(e)[:80]}'
        continue

if success_url:
    record('ucdp', 'OK',
           message=f'indirildi: {os.path.basename(success_url)}',
           files=list_files(ucdp_dir),
           size_mb=dir_size_mb(ucdp_dir),
           duration_s=time.time()-t0)
else:
    record('ucdp', 'FAIL',
           message=f'Tüm URL\'ler başarısız. Son hata: {last_error}',
           duration_s=time.time()-t0)

## 7. Polity5

In [ ]:
t0 = time.time()
polity_dir = os.path.join(RAW_DIR, 'polity')
os.makedirs(polity_dir, exist_ok=True)

# systemicpeace.org referrer kontrolü yapar
polity_headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36',
    'Referer': 'http://www.systemicpeace.org/inscrdata.html'
}

POLITY_CANDIDATES = [
    'http://www.systemicpeace.org/inscr/p5v2018.xls',
    'https://www.systemicpeace.org/inscr/p5v2018.xls',
    'http://www.systemicpeace.org/inscr/p5v2017.xls',
]

success_url = None
last_error = None
for url in POLITY_CANDIDATES:
    try:
        download_file(url, os.path.join(polity_dir, 'polity5.xls'),
                      headers=polity_headers)
        success_url = url
        break
    except Exception as e:
        last_error = f'{type(e).__name__}: {str(e)[:80]}'

if success_url:
    record('polity5', 'OK',
           message=f'indirildi: {os.path.basename(success_url)}',
           files=list_files(polity_dir),
           size_mb=dir_size_mb(polity_dir),
           duration_s=time.time()-t0)
else:
    record('polity5', 'FAIL',
           message=f'Tüm URL\'ler başarısız. Son hata: {last_error}',
           duration_s=time.time()-t0)

## 8. V-Dem — İki fallback'li otomatik indirme

**Yöntem 1:** GitHub release'inden ZIP direkt indir (en hızlı).
**Yöntem 2:** rpy2 ile vdemdata R paketini Colab'da kur ve veriyi Python'a aktar.

İlk yöntem patlarsa otomatik olarak ikinciye geçer.

In [ ]:
t0 = time.time()
vdem_dir = os.path.join(RAW_DIR, 'vdem')
os.makedirs(vdem_dir, exist_ok=True)

vdem_success = False
method_used = None
errors = []

# YÖNTEM 1: GitHub release'inden tarball indir
# vdemdata reposu V-Dem verisini .rda dosyaları olarak içerir
GITHUB_CANDIDATES = [
    'https://github.com/vdeminstitute/vdemdata/archive/refs/tags/v16.tar.gz',
    'https://github.com/vdeminstitute/vdemdata/archive/refs/heads/master.tar.gz',
    'https://github.com/vdeminstitute/vdemdata/archive/refs/heads/main.tar.gz',
]

for url in GITHUB_CANDIDATES:
    try:
        tarball_path = os.path.join(vdem_dir, 'vdemdata.tar.gz')
        download_file(url, tarball_path)
        # Aç
        import tarfile
        with tarfile.open(tarball_path, 'r:gz') as tar:
            tar.extractall(vdem_dir)
        os.remove(tarball_path)
        method_used = f'GitHub tarball ({os.path.basename(url)})'
        vdem_success = True
        break
    except Exception as e:
        errors.append(f'GitHub {url}: {type(e).__name__}')

# YÖNTEM 2: rpy2 ile R'da vdemdata kur ve veriyi CSV olarak Python'a aktar
if not vdem_success:
    try:
        print('  GitHub başarısız, rpy2 yöntemini deniyorum...')
        
        # R kurulumu kontrol
        import subprocess
        r_check = subprocess.run(['which', 'R'], capture_output=True, text=True)
        if not r_check.stdout.strip():
            !apt-get install -y r-base > /dev/null 2>&1
        
        import rpy2.robjects as ro
        from rpy2.robjects import pandas2ri
        pandas2ri.activate()
        
        # vdemdata paketini kur
        ro.r('''
        if (!require('devtools', quietly=TRUE)) install.packages('devtools', repos='https://cloud.r-project.org')
        if (!require('vdemdata', quietly=TRUE)) {
            devtools::install_github('vdeminstitute/vdemdata', upgrade='never', quiet=TRUE)
        }
        library(vdemdata)
        write.csv(vdem, '/tmp/vdem_full.csv', row.names=FALSE)
        ''')
        
        # /tmp'den hedef dizine taşı
        import shutil
        shutil.move('/tmp/vdem_full.csv', os.path.join(vdem_dir, 'vdem_full.csv'))
        method_used = 'rpy2 + vdemdata R paketi'
        vdem_success = True
    except Exception as e:
        errors.append(f'rpy2: {type(e).__name__}: {str(e)[:80]}')

if vdem_success:
    record('vdem', 'OK',
           message=f'indirildi via {method_used}',
           files=list_files(vdem_dir)[:10],  # ilk 10 dosya yeterli
           size_mb=dir_size_mb(vdem_dir),
           duration_s=time.time()-t0)
else:
    record('vdem', 'FAIL',
           message='Tüm yöntemler başarısız: ' + ' | '.join(errors[:3]),
           duration_s=time.time()-t0)

## 9. ÖZET RAPOR

Tüm kaynakların durumunu tek bakışta göster.

In [ ]:
import pandas as pd
from datetime import datetime

df = pd.DataFrame(results)

# Renkli özet tablo
print('='*80)
print('VERİ İNDİRME ÖZET RAPORU'.center(80))
print(f'Çalışma zamanı: {datetime.now().strftime("%Y-%m-%d %H:%M")}'.center(80))
print('='*80)
print()

# Status'a göre grupla
for status in ['OK', 'PARTIAL', 'FAIL']:
    subset = df[df['status'] == status]
    if len(subset) == 0:
        continue
    icon = {'OK': '✅ BAŞARILI', 'PARTIAL': '⚠️  KISMI BAŞARILI', 'FAIL': '❌ BAŞARISIZ'}[status]
    print(f'\n{icon} ({len(subset)} kaynak)')
    print('-' * 80)
    for _, row in subset.iterrows():
        print(f"  {row['source']:25s} | {row['size_mb']:>7.1f} MB | {row['duration_s']:>6.1f}s")
        print(f"  {'':25s}   {row['message']}")
        if row['files']:
            preview = ', '.join(row['files'][:3])
            extra = f' (+{len(row["files"])-3} dosya)' if len(row['files']) > 3 else ''
            print(f"  {'':25s}   📄 {preview}{extra}")

print()
print('='*80)
total_size = df['size_mb'].sum()
total_time = df['duration_s'].sum()
n_ok = (df['status'] == 'OK').sum()
n_partial = (df['status'] == 'PARTIAL').sum()
n_fail = (df['status'] == 'FAIL').sum()
print(f'TOPLAM: {n_ok} başarılı, {n_partial} kısmi, {n_fail} başarısız')
print(f'Toplam boyut: {total_size:.1f} MB | Toplam süre: {total_time:.1f}s')
print('='*80)

# Tablo halinde de bas
print('\nÖzet tablo:')
display(df[['source', 'status', 'size_mb', 'duration_s', 'message']])

# Raporu CSV olarak kaydet
report_path = os.path.join(PROJECT_DIR, f'data/download_report_{datetime.now().strftime("%Y%m%d_%H%M")}.csv')
df.to_csv(report_path, index=False)
print(f'\n📊 Detaylı rapor kaydedildi: {report_path}')

## Sonraki adım

Başarısız kaynaklar varsa:
1. **URL değişmiş olabilir** — kaynak websitesinden güncel URL'i bul, ilgili hücredeki listeye ekle
2. **Rate limit'e takıldıysa** — `time.sleep` aralığını artır
3. **404 alınıyorsa** — sürüm numarası güncellenmiş olabilir (örn. NMC-60 → NMC-61)

Tüm kaynaklar OK olursa: `02_canonical_tables.ipynb` notebook'una geç (COW kodu hizalama + normalize edip parquet olarak yaz).